In [23]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [29]:
population = pd.read_parquet("Life_tables_data/weekly_population_departements.parquet")
deaths = pd.read_parquet("Life_tables_data/deaths_weekly.parquet")
weather = pd.read_parquet("Life_tables_data/weekly_weather.parquet")

In [30]:
population = pd.read_parquet("Life_tables_data/weekly_population_departements.parquet")
deaths = pd.read_parquet("Life_tables_data/deaths_weekly.parquet")


deaths = deaths.groupby(["Year", "Week", "Age", "Departement"], as_index=False)["Deaths"].sum()

population = population[["Year", "Week", "Age", "Departement", "Total"]].rename(columns={"Total": "Population"})
data = population.merge(deaths, on=["Year", "Week", "Age", "Departement"], how="left")
data["Deaths"] = data["Deaths"].fillna(0).astype(int)
data["Population"] = data["Population"].round().astype(int)

keys = ["Departement", "Age"]
data = data[data["Age"].between(20, 85) & data["Week"].between(22, 36)].copy()

train = data[data["Year"].between(2023, 2024)].copy()
test = data[data["Year"].isin([2025, 2026])].copy()

pooled = train.groupby(keys, as_index=False).agg(D=("Deaths", "sum"), P=("Population", "sum"))
pooled["qbase"] = np.divide(pooled["D"], pooled["P"], out=np.zeros(len(pooled)), where=pooled["P"].to_numpy() > 0)
qbase_pool = pooled[keys + ["qbase"]]
